In [1]:
# Cell 1
# =============================================================================
# Load & Concatenate Baseline Normalized Datasets Across All Nodes (Nodes A–H)
# =============================================================================

import os
import pandas as pd

# Define node identifiers and root dataset directory
NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
BASE_DIR = "dataset/normalized"

# Construct file paths for all normalized train and test splits
train_files = [os.path.join(BASE_DIR, f"Node_{node}_train_normalized.csv") for node in NODES]
test_files  = [os.path.join(BASE_DIR, f"Node_{node}_test_normalized.csv")  for node in NODES]

print("Loading normalized training datasets...")
for path in train_files:
    print(f"  - {path}")

print("\nLoading normalized testing datasets...")
for path in test_files:
    print(f"  - {path}")

# Load and concatenate data across all nodes
df_train_list = [pd.read_csv(p) for p in train_files]
df_test_list  = [pd.read_csv(p) for p in test_files]

df_train = pd.concat(df_train_list, ignore_index=True)
df_test  = pd.concat(df_test_list,  ignore_index=True)

# Display dataset shapes and target distribution
print("\n=== Combined Baseline Multi-Node Datasets (Nodes A–H) ===")
print(f"Combined Training Shape: {df_train.shape}")
print(f"Combined Testing Shape:  {df_test.shape}")
print(f"Columns: {df_train.columns.tolist()}")

print("\nTarget Label Distribution (Train):")
print(df_train["Attack"].value_counts())

print("\nTarget Label Distribution (Test):")
print(df_test["Attack"].value_counts())

Loading normalized training datasets...
  - dataset/normalized/Node_A_train_normalized.csv
  - dataset/normalized/Node_B_train_normalized.csv
  - dataset/normalized/Node_C_train_normalized.csv
  - dataset/normalized/Node_D_train_normalized.csv
  - dataset/normalized/Node_E_train_normalized.csv
  - dataset/normalized/Node_F_train_normalized.csv
  - dataset/normalized/Node_G_train_normalized.csv
  - dataset/normalized/Node_H_train_normalized.csv

Loading normalized testing datasets...
  - dataset/normalized/Node_A_test_normalized.csv
  - dataset/normalized/Node_B_test_normalized.csv
  - dataset/normalized/Node_C_test_normalized.csv
  - dataset/normalized/Node_D_test_normalized.csv
  - dataset/normalized/Node_E_test_normalized.csv
  - dataset/normalized/Node_F_test_normalized.csv
  - dataset/normalized/Node_G_test_normalized.csv
  - dataset/normalized/Node_H_test_normalized.csv

=== Combined Baseline Multi-Node Datasets (Nodes A–H) ===
Combined Training Shape: (245856, 7)
Combined Testing

In [2]:
# Cell 2
# =============================================================================
# Feature Preprocessing & Baseline Random Forest Classifier Training
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

# -----------------------------------------------------------------------------
# 1. Feature Definition & One-Hot Encoding
# -----------------------------------------------------------------------------
TARGET_COL = "Attack"
FEATURE_COLS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]
NUMERIC_FEATURES = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
EXPECTED_FEATURE_ORDER = [
    "shunt_voltage", "bus_voltage_V", "current_mA", "power_mW",
    "State_idle", "State_charging"
]

# Separate features and target
X_train_raw = df_train[FEATURE_COLS].copy()
y_train_raw = df_train[TARGET_COL].copy()

X_test_raw  = df_test[FEATURE_COLS].copy()
y_test_raw  = df_test[TARGET_COL].copy()

# One-hot encode the categorical 'State' column
X_train = pd.get_dummies(X_raw_train := X_train_raw, columns=["State"], drop_first=False)
X_test  = pd.get_dummies(X_raw_test := X_test_raw,   columns=["State"], drop_first=False)

# Ensure both state dummy columns exist in both splits
for col in ["State_idle", "State_charging"]:
    if col not in X_train.columns:
        X_train[col] = 0
    if col not in X_test.columns:
        X_test[col] = 0

# Enforce consistent feature column order
X_train = X_train[EXPECTED_FEATURE_ORDER]
X_test  = X_test[EXPECTED_FEATURE_ORDER]

# Encode target categorical labels into integers
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

# -----------------------------------------------------------------------------
# 2. Standardize Numerical Features
# -----------------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_test_scaled[NUMERIC_FEATURES]  = scaler.transform(X_test[NUMERIC_FEATURES])

X_train_np = X_train_scaled.to_numpy()
X_test_np  = X_test_scaled.to_numpy()

print(f"Target classes mapped: {dict(zip(le.classes_, range(len(le.classes_))))}")
print(f"Final input features ({len(EXPECTED_FEATURE_ORDER)}): {EXPECTED_FEATURE_ORDER}")
print(f"Training samples: {X_train_np.shape[0]} | Test samples: {X_test_np.shape[0]}")

# -----------------------------------------------------------------------------
# 3. Train Baseline Random Forest Classifier
# -----------------------------------------------------------------------------
rf_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    n_jobs=-1
)

rf_final.fit(X_train_np, y_train)

print(f"\nModel training complete.")
print(f"Out-Of-Bag (OOB) Accuracy Score: {rf_final.oob_score_:.6f}")

Target classes mapped: {'Backdoor': 0, 'none': 1, 'syn-flood': 2}
Final input features (6): ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State_idle', 'State_charging']
Training samples: 245856 | Test samples: 105456

Model training complete.
Out-Of-Bag (OOB) Accuracy Score: 0.920917


In [3]:
# Cell 3
# =============================================================================
# Serialize Baseline Model and Preprocessors to Disk (.joblib)
# =============================================================================

import os
import joblib

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Export model, label encoder, and fitted standard scaler for downstream tasks
joblib.dump(rf_final, os.path.join(MODEL_DIR, "rf_final.joblib"))
joblib.dump(le,      os.path.join(MODEL_DIR, "label_encoder.joblib"))
joblib.dump(scaler,  os.path.join(MODEL_DIR, "scaler.joblib"))

print(f"Successfully saved baseline model and preprocessors to '{MODEL_DIR}':")
print("  - rf_final.joblib")
print("  - label_encoder.joblib")
print("  - scaler.joblib")

Successfully saved baseline model and preprocessors to 'models':
  - rf_final.joblib
  - label_encoder.joblib
  - scaler.joblib


In [4]:
# Cell 4
# =============================================================================
# Per-Node Evaluation (Nodes A–H) on Individual Test Sets
# =============================================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, log_loss, mean_squared_error, roc_auc_score

def prepare_node_features(df_raw, scaler_instance, expected_cols, numeric_cols):
    """
    Applies one-hot encoding, column alignment, and numeric standardization to a single node's data.
    """
    X_raw = df_raw[FEATURE_COLS].copy()
    X = pd.get_dummies(X_raw, columns=["State"], drop_first=False)
    
    for col in ["State_idle", "State_charging"]:
        if col not in X.columns:
            X[col] = 0
            
    X = X[expected_cols]
    
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler_instance.transform(X[numeric_cols])
    return X_scaled.to_numpy()

print("Evaluating Baseline RF Performance across individual node test splits:\n")

for node in NODES:
    test_path = os.path.join(BASE_DIR, f"Node_{node}_test_normalized.csv")
    df_test_node = pd.read_csv(test_path)
    
    # Prepare features and encode target labels
    X_test_node_np = prepare_node_features(df_test_node, scaler, EXPECTED_FEATURE_ORDER, NUMERIC_FEATURES)
    y_test_node = le.transform(df_test_node[TARGET_COL])
    
    # Predict classes and probabilities
    y_test_node_pred = rf_final.predict(X_test_node_np)
    y_test_node_prob = rf_final.predict_proba(X_test_node_np)
    
    # Calculate performance metrics
    macro_f1 = f1_score(y_test_node, y_test_node_pred, average="macro")
    mse_error = mean_squared_error(y_test_node, y_test_node_pred)
    loss = log_loss(y_test_node, y_test_node_prob)
    
    try:
        roc_auc = roc_auc_score(y_test_node, y_test_node_prob, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = np.nan
    
    print(f"=== Node {node} (Samples: {len(df_test_node)}) ===")
    print(f"  Macro F1-Score:        {macro_f1:.6f}")
    print(f"  Mean Squared Error:    {mse_error:.6f}")
    print(f"  Log Loss:              {loss:.6f}")
    print(f"  ROC-AUC Score:         {roc_auc:.6f}")
    
    cm = confusion_matrix(y_test_node, y_test_node_pred)
    print("  Confusion Matrix (Rows: True, Cols: Predicted):")
    print(f"  Classes: {le.classes_}")
    print(f"  {cm.tolist()}\n")

Evaluating Baseline RF Performance across individual node test splits:

=== Node A (Samples: 13182) ===
  Macro F1-Score:        0.842614
  Mean Squared Error:    0.136246
  Log Loss:              0.295517
  ROC-AUC Score:         0.957292
  Confusion Matrix (Rows: True, Cols: Predicted):
  Classes: ['Backdoor' 'none' 'syn-flood']
  [[1827, 1211, 4], [511, 5066, 0], [10, 18, 4535]]

=== Node B (Samples: 13182) ===
  Macro F1-Score:        0.842941
  Mean Squared Error:    0.130784
  Log Loss:              0.291932
  ROC-AUC Score:         0.954259
  Confusion Matrix (Rows: True, Cols: Predicted):
  Classes: ['Backdoor' 'none' 'syn-flood']
  [[1607, 925, 3], [745, 5844, 2], [9, 4, 4043]]

=== Node C (Samples: 13182) ===
  Macro F1-Score:        0.847072
  Mean Squared Error:    0.121453
  Log Loss:              0.290012
  ROC-AUC Score:         0.953050
  Confusion Matrix (Rows: True, Cols: Predicted):
  Classes: ['Backdoor' 'none' 'syn-flood']
  [[1466, 1067, 2], [505, 6593, 0], [3, 9,

In [7]:
#SUMMARY#
txt = '''
Summary of Changes in Notebook 03:
- Cell Consolidation & Re-ordering: Placed artifact saving (Cell 3) immediately after training (Cell 2) to ensure files are saved before running evaluation.
- Removed Redundant Cells: Deleted empty placeholder and exploratory comment cells.
- Unified Preprocessing Helper: Streamlined prepare_node_features to eliminate code duplication.
- Standardized Configuration: Used uppercase constants (NODES, FEATURE_COLS, MODEL_DIR, etc.).  
'''
print(txt)


Summary of Changes in Notebook 03:
- Cell Consolidation & Re-ordering: Placed artifact saving (Cell 3) immediately after training (Cell 2) to ensure files are saved before running evaluation.
- Removed Redundant Cells: Deleted empty placeholder and exploratory comment cells.
- Unified Preprocessing Helper: Streamlined prepare_node_features to eliminate code duplication.
- Standardized Configuration: Used uppercase constants (NODES, FEATURE_COLS, MODEL_DIR, etc.).  

